# Spike 05 — Retrieval Quality Check

**Goal:** Ask real questions against the indexed documents and evaluate whether the answers are correct.

**Time box:** 1-2 hours

**Question to answer:** Does the end-to-end pipeline (OCR → PageIndex → LLM) return accurate, well-sourced answers?

**Done when:** You have a scored evaluation table showing which questions the system answered correctly.

---

### What we're testing

This spike closes the loop: we go from raw question → retrieved context → LLM answer → human judgment.  
The output is not production code — it's a **decision**: is the pipeline good enough to build on?

We use Groq (Llama 3.3 70B) as the answer-generation LLM — fast, free, and we already have it set up.

## Step 1 — Setup

In [ ]:
import requests
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(dotenv_path="../.env")

# PageIndex — retrieval
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
BASE_URL          = "https://api.pageindex.ai/v1"   # adjust as needed
PI_HEADERS        = {
    "Authorization": f"Bearer {PAGEINDEX_API_KEY}",
    "Content-Type" : "application/json"
}

# Groq — answer generation
groq_client = OpenAI(
    api_key  = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)
MODEL = "llama-3.3-70b-versatile"

print("✅ PageIndex client configured")
print(f"✅ Groq client ready — model: {MODEL}")

## Step 2 — Retrieval Function

Send a question to PageIndex and get back the most relevant passages.

In [ ]:
def retrieve(question: str, language: str = "en", top_k: int = 5) -> list:
    """
    Query PageIndex and return the top_k most relevant passages.
    Returns a list of dicts with 'content' and 'source' keys.
    """
    payload = {
        "query"   : question,
        "language": language,
        "top_k"   : top_k
    }

    r = requests.post(
        f"{BASE_URL}/query",
        headers = PI_HEADERS,
        json    = payload,
        timeout = 30
    )

    if r.status_code != 200:
        print(f"❌ Retrieval failed: {r.status_code} — {r.text[:300]}")
        return []

    data = r.json()
    # Adjust these keys based on PageIndex's actual response schema
    return data.get("results", data.get("passages", []))


# Quick test
test_results = retrieve("What are the livability indicators for Madinah?", language="en")
print(f"Retrieved {len(test_results)} passages")
if test_results:
    print("\nFirst passage preview:")
    first = test_results[0]
    print(json.dumps(first, indent=2, ensure_ascii=False)[:500])

## Step 3 — Answer Generation Function

Take retrieved passages + question → send to Groq → get a grounded answer with citations.

In [ ]:
def generate_answer(question: str, passages: list, language: str = "en") -> str:
    """
    Generate a grounded answer from retrieved passages.
    The LLM is instructed to only use what's in the context — no hallucination.
    """
    if not passages:
        return "No relevant passages found to answer this question."

    # Build context block from retrieved passages
    context_parts = []
    for i, p in enumerate(passages, 1):
        # Try common key names for content
        content = p.get("content") or p.get("text") or p.get("passage") or str(p)
        source  = p.get("source") or p.get("doc_id") or p.get("id") or f"passage_{i}"
        context_parts.append(f"[Source {i}: {source}]\n{content}")

    context = "\n\n".join(context_parts)

    # Language-aware system prompt
    if language == "ar":
        system = (
            "أنت مساعد متخصص في تحليل تقارير التنمية الحضرية لمدينة المدينة المنورة. "
            "أجب على الأسئلة بالعربية فقط باستخدام المعلومات المقدمة. "
            "اذكر المصدر لكل معلومة. إذا لم تجد الإجابة في النصوص المقدمة، قل ذلك صراحة."
        )
    else:
        system = (
            "You are an expert analyst of Madinah urban development reports. "
            "Answer questions using ONLY the provided context passages. "
            "Cite the source number for each fact (e.g. [Source 1]). "
            "If the answer is not in the context, say so explicitly — do NOT guess."
        )

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"Context passages:\n\n{context}\n\nQuestion: {question}"}
    ]

    response = groq_client.chat.completions.create(
        model      = MODEL,
        messages   = messages,
        max_tokens = 1024
    )

    return response.choices[0].message.content


print("✅ generate_answer() defined")

## Step 4 — Full Pipeline Function

In [ ]:
def ask(question: str, language: str = "en", verbose: bool = True) -> dict:
    """
    Full pipeline: question → retrieve → generate → return answer with sources.
    """
    if verbose:
        print(f"Q: {question}")
        print("-" * 60)

    # Step 1: Retrieve relevant passages
    passages = retrieve(question, language=language)
    if verbose:
        print(f"Retrieved {len(passages)} passages from PageIndex")

    # Step 2: Generate answer
    answer = generate_answer(question, passages, language=language)

    if verbose:
        print(f"\nAnswer:\n{answer}")
        print()

    return {
        "question": question,
        "answer"  : answer,
        "sources" : [p.get("source", "") for p in passages]
    }


print("✅ ask() pipeline ready")

## Step 5 — Run the Evaluation Test Set

These are real questions a policy analyst or urban planner might ask about the Madinah report.  
After each answer, we score it manually: ✅ correct / ⚠️ partial / ❌ wrong.

In [ ]:
# Evaluation questions — (question, language)
TEST_QUESTIONS = [
    # English
    ("What are the livability indicators used in this report?",            "en"),
    ("What is the population of Madinah according to the report?",         "en"),
    ("What sustainability goals were achieved in 2024?",                   "en"),
    ("Which neighborhoods scored highest on livability?",                  "en"),
    ("What are the key challenges facing Madinah's urban development?",    "en"),

    # Arabic
    ("ما هي مؤشرات قابلية العيش المستخدمة في هذا التقرير؟",              "ar"),
    ("ما هو عدد سكان المدينة المنورة وفقاً للتقرير؟",                     "ar"),
    ("ما هي أبرز إنجازات التنمية المستدامة في عام 2024؟",                 "ar"),
]

results = []

for i, (question, lang) in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*65}")
    print(f"Test {i+1}/{len(TEST_QUESTIONS)} [{lang.upper()}]")
    result = ask(question, language=lang, verbose=True)
    results.append(result)

## Step 6 — Score the Results

Run this cell after reading each answer. Score manually — you know the content of the reports.

In [ ]:
# Fill in your scores after reading each answer above
# Options: 'correct', 'partial', 'wrong', 'unsure'
SCORES = [
    'unsure',  # Q1 EN
    'unsure',  # Q2 EN
    'unsure',  # Q3 EN
    'unsure',  # Q4 EN
    'unsure',  # Q5 EN
    'unsure',  # Q1 AR
    'unsure',  # Q2 AR
    'unsure',  # Q3 AR
]

# Summary
from collections import Counter
score_counts = Counter(SCORES)

total    = len(SCORES)
correct  = score_counts['correct']
partial  = score_counts['partial']
wrong    = score_counts['wrong']
unsure   = score_counts['unsure']

print("=" * 50)
print("SPIKE 05 — EVALUATION RESULTS")
print("=" * 50)
print(f"Total questions : {total}")
print(f"✅ Correct      : {correct}  ({correct/total*100:.0f}%)")
print(f"⚠️  Partial      : {partial}  ({partial/total*100:.0f}%)")
print(f"❌ Wrong        : {wrong}   ({wrong/total*100:.0f}%)")
print(f"❓ Unsure       : {unsure}   (score these manually)")
print()

effective_score = (correct + 0.5 * partial) / (total - unsure) * 100 if (total - unsure) > 0 else 0
print(f"Effective score : {effective_score:.0f}%")
print()

if effective_score >= 70:
    print("✅ SPIKE PASSED — Pipeline is good enough to build on")
    print("   Next: Apply Martin Fowler patterns (query rewriting, re-ranking, guards)")
elif effective_score >= 40:
    print("⚠️  SPIKE PARTIAL — Investigate failures before continuing")
    print("   Common fixes: improve OCR prompt, re-index with better chunking")
else:
    print("❌ SPIKE FAILED — Pipeline needs significant work")
    print("   Check: Did Spike 02/03 produce clean text? Is PageIndex indexing correctly?")

## Step 7 — Diagnose Failures

If any answers were wrong, investigate here.

In [ ]:
# Pick any question index (0-based) to debug
DEBUG_IDX = 0

if DEBUG_IDX < len(results):
    r = results[DEBUG_IDX]
    q, lang = TEST_QUESTIONS[DEBUG_IDX]

    print(f"=== DEBUGGING Q{DEBUG_IDX+1} ===")
    print(f"Question : {q}")
    print(f"Language : {lang}")
    print()
    print("Raw passages retrieved:")

    passages = retrieve(q, language=lang)
    for i, p in enumerate(passages, 1):
        print(f"\n--- Passage {i} ---")
        content = p.get("content") or p.get("text") or str(p)
        source  = p.get("source") or p.get("doc_id") or "unknown"
        print(f"Source  : {source}")
        print(f"Content : {content[:500]}")

    # Diagnosis hints:
    print("\n=== DIAGNOSIS ===")
    if not passages:
        print("❌ No passages returned — PageIndex may not be indexing correctly")
    elif all(len(p.get('content','')) < 50 for p in passages):
        print("⚠️  Passages are very short — OCR quality issue? Check ocr_output/ files")
    else:
        print("Passages look OK — the problem may be in the answer generation prompt")
        print("Try adjusting the system prompt in generate_answer()")